In [2]:
here::i_am("snakemake/01_create_arrow.R")
library(BSgenome.Ocuniculus.NCBI.oryCun2)

# I/O
io$output.directory <- file.path(io$basedir,"ArchR")
dir.create(file.path(io$output.directory), showWarnings = FALSE)
source(here::here("settings.R"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC_final

Setting default number of Parallel threads to 1.



In [3]:
# ArchR options
addArchRThreads(threads = 12)

Setting default number of Parallel threads to 12.



In [4]:
geneAnnotation = readRDS(file.path(io$basedir, 'geneAnnotation_new.rds'))

In [5]:
genomeAnnotation = createGenomeAnnotation(
  genome = BSgenome.Ocuniculus.NCBI.oryCun2,
  chromSizes = NULL,
  blacklist = NULL,
  filter = FALSE,
  filterChr = c("chrM")
)


Getting genome..

Getting chromSizes..

Getting blacklist..

Blacklist not downloaded! Continuing without, be careful for downstream biases..



In [6]:
seqlevelsStyle(genomeAnnotation$chromSizes) #<- seqstyle
seqlevelsStyle(geneAnnotation$genes) #<- seqstyle
seqlevelsStyle(geneAnnotation$TSS)# <- seqstyle
seqlevelsStyle(geneAnnotation$exons) #<- seqstyle
seqlevelsStyle(BSgenome.Ocuniculus.NCBI.oryCun2) #<- seqstyle

[1] "UCSC"

[1] "UCSC"

[1] "UCSC"

[1] "UCSC"

[1] "UCSC"

In [7]:
args = list()
args$sample = 'BGRGP5'
args$min_fragments = 2500
args$min_tss_score = 2
args$outdir = io$output.directory

In [8]:
fragment_file_path = paste0(io$basedir, '/data/', args$sample, '_fragments.tsv.gz')

In [ ]:
#Create Arrow File
ArrowFiles <- createArrowFiles(
  inputFiles = fragment_file_path,
  sampleNames = paste0('rabbit_', args$sample),
    
  minTSS = args$min_tss_score, #Dont set this too high because you can always increase later
  minFrags = args$min_fragments , 
    
  addTileMat = TRUE,
  addGeneScoreMat = FALSE,
    
  geneAnnotation = geneAnnotation,
  genomeAnnotation = genomeAnnotation
)

ArchR logging to : ArchRLogs/ArchR-createArrows-2de0017f59672-Date-2022-01-26_Time-10-09-43.log
If there is an issue, please report to github with logFile!

Cleaning Temporary Files

2022-01-26 10:09:43 : Batch Execution w/ safelapply!, 0 mins elapsed.

(rabbit_BGRGP5 : 1 of 1) Determining Arrow Method to use!

Attempting to index /rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC_final/data/BGRGP5_fragments.tsv.gz as tabix..

2022-01-26 10:11:53 : (rabbit_BGRGP5 : 1 of 1) Reading In Fragments from inputFiles (readMethod = tabix), 2.169 mins elapsed.

2022-01-26 10:11:53 : (rabbit_BGRGP5 : 1 of 1) Tabix Bed To Temporary File, 2.169 mins elapsed.

2022-01-26 10:34:12 : (rabbit_BGRGP5 : 1 of 1) Successful creation of Temporary File, 24.488 mins elapsed.

2022-01-26 10:34:12 : (rabbit_BGRGP5 : 1 of 1) Creating ArrowFile From Temporary File, 24.488 mins elapsed.



In [ ]:
ArrowFiles = list.files(io$output.directory, pattern ='arrow')

In [ ]:
proj <- ArchRProject(
  ArrowFiles = ArrowFiles, 
  outputDirectory = "Project",
  copyArrows = TRUE, #This is recommened so that you maintain an unaltered copy for later usage.
  geneAnnotation = geneAnnotation,
  genomeAnnotation = genomeAnnotation
)

In [ ]:
proj <- addMotifAnnotations(ArchRProj = proj, motifSet = "cisbp", name = "Motif")

In [ ]:
proj <- addBgdPeaks(proj)

In [ ]:
proj <- addDeviationsMatrix(
  ArchRProj = proj, 
  peakAnnotation = "Motif",
  force = TRUE
)

In [ ]:
plotVarDev <- getVarDeviations(proj, name = "MotifMatrix", plot = TRUE)

In [ ]:
plotVarDev